# Quick Maths with Matrices!
---

<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Quick-Maths-with-Matrices!" data-toc-modified-id="Quick-Maths-with-Matrices!-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Quick Maths with Matrices!</a></span><ul class="toc-item"><li><span><a href="#Import-Libraries" data-toc-modified-id="Import-Libraries-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Import Libraries</a></span></li><li><span><a href="#Test-Framework" data-toc-modified-id="Test-Framework-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Test Framework</a></span></li></ul></li><li><span><a href="#Optimizing-Matrix-Multiplications" data-toc-modified-id="Optimizing-Matrix-Multiplications-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Optimizing Matrix Multiplications</a></span><ul class="toc-item"><li><span><a href="#For-Loop" data-toc-modified-id="For-Loop-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>For Loop</a></span></li><li><span><a href="#Array-Slicing" data-toc-modified-id="Array-Slicing-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>Array Slicing</a></span></li><li><span><a href="#Improvement-with-array-slicing" data-toc-modified-id="Improvement-with-array-slicing-2.3"><span class="toc-item-num">2.3&nbsp;&nbsp;</span>Improvement with array slicing</a></span></li><li><span><a href="#Array-Broadcasting" data-toc-modified-id="Array-Broadcasting-2.4"><span class="toc-item-num">2.4&nbsp;&nbsp;</span>Array Broadcasting</a></span></li><li><span><a href="#Improvement-with-array-broadcasting" data-toc-modified-id="Improvement-with-array-broadcasting-2.5"><span class="toc-item-num">2.5&nbsp;&nbsp;</span>Improvement with array broadcasting</a></span></li><li><span><a href="#Einstein-Sum" data-toc-modified-id="Einstein-Sum-2.6"><span class="toc-item-num">2.6&nbsp;&nbsp;</span>Einstein Sum</a></span></li><li><span><a href="#Improvement-with-einstein-sum" data-toc-modified-id="Improvement-with-einstein-sum-2.7"><span class="toc-item-num">2.7&nbsp;&nbsp;</span>Improvement with einstein sum</a></span></li><li><span><a href="#Linear-Algebra-Libraries" data-toc-modified-id="Linear-Algebra-Libraries-2.8"><span class="toc-item-num">2.8&nbsp;&nbsp;</span>Linear Algebra Libraries</a></span></li><li><span><a href="#Improvement-with-linear-algebra-libraries" data-toc-modified-id="Improvement-with-linear-algebra-libraries-2.9"><span class="toc-item-num">2.9&nbsp;&nbsp;</span>Improvement with linear algebra libraries</a></span></li></ul></li></ul></div>

## Import Libraries

In [1]:
import torch
import timeit
import operator
from functools import partial

## Test Framework

In [2]:
def test(a, b, compare, compare_name=None):
    if compare_name is None:
        compare_name = compare.__name__
    assert compare(a, b),\
    f"{compare_name} check failed:\n{a}\n{b}"

def test_equality(a, b):
    test(a, b, operator.eq, "Equality")

def test_approximately(a, b):
    allclose = partial(torch.allclose, atol=1e-5, rtol=1e-03)
    if not isinstance(a, torch.Tensor) or not isinstance(b, torch.Tensor):
        a = torch.tensor(a)
        b = torch.tensor(b)
    test(a, b, allclose, "Approximate Equality")

In [3]:
test_equality(1e-5,1e-5)

In [4]:
test_approximately(1e-5, 1e-6)

# Optimizing Matrix Multiplications

**Test Variables**

In [5]:
A = torch.randn([10,10])
B = torch.randn([10,10])

In [6]:
(A@B).shape

torch.Size([10, 10])

## For Loop

In [7]:
def matmul(A,B):
    A_rows, A_cols = A.shape
    B_rows, B_cols = B.shape
    assert A_cols==B_rows,\
    f"Inner dimensions must match: {A_cols} not equal to {B_rows}"
    C = torch.zeros([A_rows, B_cols])
    for i in range(A_rows):
        for j in range(B_cols):
            for k in range(A_cols):
                C[i,j] += A[i,k] * B[k,j]
    return C

In [8]:
matmul(A,B)

tensor([[-1.1746, -0.6003,  4.5133,  0.5549, -1.3951,  8.2522, -3.0070, -1.6351,
          3.2492, -0.4468],
        [ 0.6406,  9.1082, -4.0124, -2.1507, -1.9986, -7.8985, -0.6630, -0.4537,
          6.3365,  6.1809],
        [-2.6682, -6.8922,  0.9535,  7.0017,  1.3691,  4.1847, -0.1249, -0.2006,
         -1.1729, -5.3074],
        [ 4.4569,  0.1378,  0.5874, -5.0908,  0.9787,  1.2787, -3.9773,  3.4696,
         -1.0233, -2.0289],
        [-0.8707,  1.9152, -1.7778,  0.5442,  0.7938, -3.3847,  1.6799,  1.0061,
          1.8372, -1.2610],
        [ 2.6344, -0.2452,  2.0803, -2.6766, -3.4628, -2.4018, -4.1409,  2.1538,
         -0.0997, -0.1876],
        [ 3.3106, -0.3195,  0.4233, -0.0944, -0.7465, -2.4804, -0.6526, -0.9198,
          0.1968, -1.6679],
        [ 5.8002, -1.8782, -5.7302,  3.9666,  2.8957, -5.3170, -2.4726, -1.7825,
         -0.8461, -3.8210],
        [ 0.7135,  0.1271, -4.1076, -0.6258,  3.0679,  2.0919, -4.3802,  2.4360,
         -2.1934, -1.4536],
        [ 0.2629,  

In [9]:
test_approximately(matmul(A, B), (A@B))

In [10]:
matmul_loop_time = timeit.timeit(partial(matmul,A,B), number=10)
matmul_loop_time

0.21082532200011883

In [11]:
# Call the matrix multiplication function
C = matmul(A, B)
print("Result Matrix C (A x B):")
print(C)

Result Matrix C (A x B):
tensor([[-1.1746, -0.6003,  4.5133,  0.5549, -1.3951,  8.2522, -3.0070, -1.6351,
          3.2492, -0.4468],
        [ 0.6406,  9.1082, -4.0124, -2.1507, -1.9986, -7.8985, -0.6630, -0.4537,
          6.3365,  6.1809],
        [-2.6682, -6.8922,  0.9535,  7.0017,  1.3691,  4.1847, -0.1249, -0.2006,
         -1.1729, -5.3074],
        [ 4.4569,  0.1378,  0.5874, -5.0908,  0.9787,  1.2787, -3.9773,  3.4696,
         -1.0233, -2.0289],
        [-0.8707,  1.9152, -1.7778,  0.5442,  0.7938, -3.3847,  1.6799,  1.0061,
          1.8372, -1.2610],
        [ 2.6344, -0.2452,  2.0803, -2.6766, -3.4628, -2.4018, -4.1409,  2.1538,
         -0.0997, -0.1876],
        [ 3.3106, -0.3195,  0.4233, -0.0944, -0.7465, -2.4804, -0.6526, -0.9198,
          0.1968, -1.6679],
        [ 5.8002, -1.8782, -5.7302,  3.9666,  2.8957, -5.3170, -2.4726, -1.7825,
         -0.8461, -3.8210],
        [ 0.7135,  0.1271, -4.1076, -0.6258,  3.0679,  2.0919, -4.3802,  2.4360,
         -2.1934, -1.4

In [12]:
matmul_time = timeit.timeit(partial(matmul, A, B), number=1)
print(f"Time taken for matmul(A, B)A100GPU {matmul_time:.6f} seconds")

Time taken for matmul(A, B)A100GPU 0.021845 seconds
